In [ ]:
# ============================================
# CELL 1: Check GPU and System Info
# ============================================
!nvidia-smi
print("\n" + "="*50)
print("GPU is available!" if __import__('torch').cuda.is_available() else "⚠️ WARNING: No GPU detected!")
print("="*50)

In [ ]:
# ============================================
# CELL 2: Install System Dependencies
# ============================================
print("📦 Installing system dependencies...")
!apt-get update -qq
!apt-get install -y -qq redis-server ffmpeg
print("✅ System dependencies installed!")

In [ ]:
# ============================================
# CELL 3: Clone VIDRA Project from GitHub
# ============================================
import os

print("📁 Setting up VIDRA project...\n")

# Check if already cloned
if os.path.exists('/content/Vidra-Your-AI-Video-Director'):
    print("✅ VIDRA project already exists!")
    %cd /content/Vidra-Your-AI-Video-Director
else:
    # Clone from GitHub
    print("📥 Cloning VIDRA from GitHub...\n")
    !git clone https://github.com/Safwan2003/Vidra-Your-AI-Video-Director.git
    
    # Check if clone was successful
    if os.path.exists('/content/Vidra-Your-AI-Video-Director'):
        %cd /content/Vidra-Your-AI-Video-Director
        print("\n✅ VIDRA project cloned successfully!")
        print(f"📂 Current directory: {os.getcwd()}")
    else:
        print("❌ Error: Failed to clone repository")
        print("Please check your internet connection and try again")

In [ ]:
# ============================================
# CELL 4: Install Python Dependencies
# ============================================
print("📦 Installing Python packages (this may take 5-10 minutes)...\n")

# Install backend dependencies
print("Installing backend packages...")
!pip install -q -r backend/requirements.txt

# Install TTS separately (compatible version for Colab)
print("\nInstalling TTS (Text-to-Speech)...")
!pip install -q git+https://github.com/coqui-ai/TTS@dev

# Install frontend dependencies
print("\nInstalling frontend packages...")
!pip install -q -r frontend/requirements.txt

# Install additional tools
print("\nInstalling additional tools (pyngrok, dotenv, cloudflared)...")
!pip install -q pyngrok python-dotenv cloudflared

print("\n✅ All dependencies installed!")

In [ ]:
# ============================================
# CELL 5: Configure Environment Variables
# ============================================
import os

print("⚙️ Configuring environment...\n")

# Set your API key here (REQUIRED for storyboard generation)
# Get FREE key from: https://console.groq.com
API_KEY = ""  # ⬅️ PASTE YOUR GROQ API KEY HERE

if not API_KEY:
    print("⚠️ WARNING: No API key set!")
    print("Get a FREE key from: https://console.groq.com")
    print("Then paste it in the API_KEY variable above and re-run this cell.")
    print("\nContinuing with mock storyboard generation...\n")
else:
    os.environ['GROQ_API_KEY'] = API_KEY
    print("✅ API key configured!\n")

# Optional: ngrok token for public URL (used in Cell 9)
NGROK_AUTHTOKEN = ""  # Optional: paste your ngrok token here to use ngrok
if NGROK_AUTHTOKEN:
    os.environ['NGROK_AUTHTOKEN'] = NGROK_AUTHTOKEN
    print("✅ ngrok authtoken configured!\n")
else:
    print("ℹ️ No ngrok token set; Cell 9 will use Cloudflared fallback.\n")

# Set other environment variables
os.environ['REDIS_URL'] = 'redis://localhost:6379/0'
os.environ['OUTPUT_DIR'] = './outputs'
os.environ['TORCH_DTYPE'] = 'float16'

# Create output directories
!mkdir -p outputs/images outputs/videos outputs/audio outputs/final

print("✅ Environment configured!")

In [ ]:
# ============================================
# CELL 6: Start Redis Server
# ============================================
import subprocess
import time

print("🚀 Starting Redis server...")
redis_process = subprocess.Popen(
    ['redis-server', '--daemonize', 'yes'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(2)

# Test Redis connection
import redis
try:
    r = redis.Redis(host='localhost', port=6379, db=0)
    r.ping()
    print("✅ Redis is running!")
except Exception as e:
    print(f"❌ Redis connection failed: {e}")

In [ ]:
# ============================================
# CELL 7: Start Celery Worker (Background)
# ============================================
import subprocess
import time
import os

print("🚀 Starting Celery worker...")

# Use current working directory or GitHub repo path
project_dir = os.getcwd() if 'Vidra-Your-AI-Video-Director' in os.getcwd() else '/content/Vidra-Your-AI-Video-Director'

celery_process = subprocess.Popen(
    ['celery', '-A', 'backend.main.celery_app', 'worker', '--loglevel=info', '--pool=solo'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    cwd=project_dir
)
time.sleep(5)
print("✅ Celery worker started!")
print(f"   Working directory: {project_dir}")
print("   (Processing video generation tasks in background)")

In [ ]:
# ============================================
# CELL 8: Start FastAPI Backend (Background)
# ============================================
import subprocess
import time
import os
import requests

print("🚀 Starting FastAPI backend...")

# Use current working directory or GitHub repo path
project_dir = os.getcwd() if 'Vidra-Your-AI-Video-Director' in os.getcwd() else '/content/Vidra-Your-AI-Video-Director'

fastapi_process = subprocess.Popen(
    ['uvicorn', 'backend.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    cwd=project_dir
)

# Wait longer for FastAPI to start and check for errors
print("   Waiting for FastAPI to initialize...")
time.sleep(10)

# Check if process is still running
if fastapi_process.poll() is not None:
    # Process died, get error output
    stdout, stderr = fastapi_process.communicate()
    print("❌ FastAPI failed to start!")
    print("\n📋 Error output:")
    if stderr:
        print(stderr.decode('utf-8'))
    if stdout:
        print(stdout.decode('utf-8'))
else:
    print("✅ FastAPI backend running on http://localhost:8000")
    print(f"   Working directory: {project_dir}")
    
    # Test backend
    max_retries = 5
    for i in range(max_retries):
        try:
            response = requests.get('http://localhost:8000/', timeout=2)
            print(f"   ✅ Backend responding: {response.json()}")
            break
        except Exception as e:
            if i < max_retries - 1:
                print(f"   ⏳ Attempt {i+1}/{max_retries} failed, retrying...")
                time.sleep(2)
            else:
                print(f"   ⚠️ Backend not responding after {max_retries} attempts: {e}")
                print("   💡 Tip: Check if Redis and Celery are running in previous cells")

In [ ]:
# ============================================
# CELL 9: Start Streamlit with Public URL
# ============================================
import subprocess
import time
import os
import re

print("🚀 Starting Streamlit UI...\n")

# Use current working directory or GitHub repo path
project_dir = os.getcwd() if 'Vidra-Your-AI-Video-Director' in os.getcwd() else '/content/Vidra-Your-AI-Video-Director'

# Start Streamlit in background
streamlit_process = subprocess.Popen(
    ['streamlit', 'run', 'frontend/streamlit_app.py', '--server.port', '8501', '--server.headless', 'true'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    cwd=project_dir
)
# Give Streamlit time to boot
time.sleep(8)

# Create public URL: prefer ngrok if token is set, otherwise fallback to Cloudflared
print("🌐 Creating public URL...\n")
public_url = None

ngrok_token = os.environ.get('NGROK_AUTHTOKEN')
if ngrok_token:
    try:
        from pyngrok import ngrok
        ngrok.set_auth_token(ngrok_token)
        tunnel = ngrok.connect(8501)
        public_url = str(tunnel)
        print("✅ ngrok tunnel established")
    except Exception as e:
        print(f"⚠️ ngrok failed: {e}")
        print("🔁 Falling back to Cloudflared...")

if not public_url:
    try:
        # Launch Cloudflared tunnel via CLI and parse the URL from logs
        cf_process = subprocess.Popen(
            ['cloudflared', 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            cwd=project_dir,
            text=True,
            bufsize=1
        )
        start = time.time()
        while time.time() - start < 30:
            line = cf_process.stdout.readline()
            if not line:
                time.sleep(0.2)
                continue
            # Look for a public URL in the output (usually *.trycloudflare.com)
            m = re.search(r'https?://[\w\.-]+\.trycloudflare\.com\S*', line)
            if m:
                public_url = m.group(0)
                break
        if not public_url:
            raise RuntimeError("Cloudflared did not provide a public URL in time.")
        print("✅ Cloudflared tunnel established")
    except Exception as e:
        print(f"❌ Failed to create public URL: {e}")
        print("   Tip: Set NGROK_AUTHTOKEN in Cell 5 and re-run this cell.")
        raise

print("="*60)
print("🎉 VIDRA IS READY!")
print("="*60)
print(f"\n🔗 Access VIDRA at: {public_url}\n")
print("📝 Click the link above to open the UI in a new tab")
print("\n💡 Tips:")
print("   - First video will take 5-10 minutes (models need to load)")
print("   - Subsequent videos are faster")
print("   - Keep this notebook running while generating videos")
print("   - GPU session lasts 12 hours (free tier)")
print("\n⚡ Quick test prompt:")
print("   'A peaceful sunrise over mountains with birds flying'")
print("="*60)

In [ ]:
# ============================================
# CELL 10: Monitor Logs (Optional)
# ============================================
# Run this cell to see real-time logs from Celery worker
# Useful for debugging

print("📊 Monitoring Celery worker logs...")
print("Press STOP button to exit monitoring\n")
print("="*60)

!tail -f /content/vidra/celery.log 2>/dev/null || echo "No logs yet. Generate a video to see logs."

In [ ]:
# ============================================
# CELL 11: View Generated Files (Optional)
# ============================================
# Run this to see all generated videos and assets

import os
from IPython.display import Video, Audio, Image, display

print("📁 Generated Files:\n")

# List final videos
project_dir = os.getcwd() if 'Vidra-Your-AI-Video-Director' in os.getcwd() else '/content/Vidra-Your-AI-Video-Director'
final_dir = os.path.join(project_dir, 'outputs/final')
if os.path.exists(final_dir):
    videos = [f for f in os.listdir(final_dir) if f.endswith('.mp4')]
    if videos:
        print(f"✅ {len(videos)} video(s) generated:\n")
        for vid in videos:
            vid_path = os.path.join(final_dir, vid)
            print(f"   📹 {vid}")
            # Display latest video
        if videos:
            latest = os.path.join(final_dir, sorted(videos)[-1])
            print(f"\n🎬 Latest video preview:")
            display(Video(latest, width=800))
    else:
        print("   No videos generated yet.")
else:
    print("   No output directory found.")

In [ ]:
# ============================================
# CELL 12: Download Generated Video (Optional)
# ============================================
# Run this to download the latest video to your computer

import os
from google.colab import files

project_dir = os.getcwd() if 'Vidra-Your-AI-Video-Director' in os.getcwd() else '/content/Vidra-Your-AI-Video-Director'
final_dir = os.path.join(project_dir, 'outputs/final')
if os.path.exists(final_dir):
    videos = [f for f in os.listdir(final_dir) if f.endswith('.mp4')]
    if videos:
        latest = os.path.join(final_dir, sorted(videos)[-1])
        print(f"📥 Downloading: {os.path.basename(latest)}")
        files.download(latest)
        print("✅ Download complete!")
    else:
        print("❌ No videos to download. Generate a video first!")
else:
    print("❌ Output directory not found.")

In [ ]:
# ============================================
# CELL 13: Stop All Services (Cleanup)
# ============================================
# Run this when you're done to free up resources

import subprocess

print("🛑 Stopping all services...\n")

# Kill processes
!pkill -f streamlit
!pkill -f uvicorn
!pkill -f celery
!pkill -f redis-server

print("✅ All services stopped!")
print("\n💡 To restart, run cells 6-9 again.")

---

## 🎯 Troubleshooting

### Problem: "No GPU detected"
**Solution**: Go to Runtime → Change runtime type → Select T4 GPU → Save

### Problem: "Out of memory"
**Solution**: 
- Use shorter videos (30s instead of 60s)
- Restart runtime and run again
- Upgrade to Colab Pro for more RAM

### Problem: "Models taking too long to load"
**Solution**: First run always takes 5-10 minutes. Subsequent runs are faster (models cached).

### Problem: "Streamlit not loading"
**Solution**: 
- Wait 10-15 seconds after running Cell 9
- Check if ngrok URL is correct
- Re-run Cell 9

### Problem: "Video generation fails"
**Solution**: 
- Check Cell 10 for error logs
- Make sure API key is set in Cell 5
- Ensure GPU is enabled

---

## 📊 Performance Notes

**Expected Generation Times** (on free T4 GPU):
- Storyboard: 5-10 seconds
- Images (4 scenes): 2-3 minutes
- Animation (4 scenes): 5-8 minutes
- Audio: 30 seconds
- Final edit: 20 seconds

**Total: 8-12 minutes for a 30-second video**

---

## 💰 Cost Analysis

| Resource | Cost |
|----------|------|
| Google Colab GPU | FREE (12h sessions) |
| Groq API (Llama 3.1) | FREE |
| Storage (Colab) | FREE (temporary) |
| All AI Models | FREE (open-source) |
| **Total MVP Cost** | **$0** |

---

## 🚀 Next Steps

1. ✅ Test with different prompts and styles
2. ✅ Experiment with parameters (duration, style)
3. ✅ Share ngrok URL with team for testing
4. 🔄 For production: Deploy to RunPod/Vast.ai
5. 🔄 Add cloud storage (R2/S3) for persistence
6. 🔄 Optimize model loading and caching

---

**Made with ❤️ by the VIDRA team**